<a href="https://colab.research.google.com/github/quyetttcoder/Fine-tune-LLM-with-small-data/blob/main/Fine_tune_Llama_31_8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q unsloth datasets trl accelerate bitsandbytes tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [ ]:
import json
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from sklearn.metrics import f1_score
from tqdm import tqdm

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1536,
    dtype=torch.float16,
    load_in_4bit=True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    base_model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.7.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from unsloth.chat_templates import get_chat_template


tokenizer = get_chat_template(
    base_tokenizer,
    chat_template = "llama-3.1",
)

In [ ]:
def formatting_prompts_func(examples):
    conversaions = []

    for q, a in zip(examples["question"], examples["answer"]):

        conversaions.append([
            {"role": "system", "content": "You are an admissions consultant."},
            {"role": "user", "content": str(q)},
            {"role": "assistant", "content": str(a)},
        ])

    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False
    ) for convo in conversaions]

    return {"text": texts}

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from datasets import load_dataset
dataset = load_dataset("quyetdev/QA_Admission_dataset")

Repo card metadata block was not found. Setting CardData to empty.


train_dataset.csv:   0%|          | 0.00/767k [00:00<?, ?B/s]

valid_dataset.csv:   0%|          | 0.00/86.7k [00:00<?, ?B/s]

test_dataset.csv:   0%|          | 0.00/83.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/935 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/117 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.filter(
    lambda x: x["question"] is not None and x["answer"] is not None
)

Filter:   0%|          | 0/935 [00:00<?, ? examples/s]

Filter:   0%|          | 0/117 [00:00<?, ? examples/s]

Filter:   0%|          | 0/117 [00:00<?, ? examples/s]

In [ ]:
train_ds = dataset["train"].map(formatting_prompts_func, batched=True)
valid_ds = dataset["validation"].map(formatting_prompts_func, batched=True)
test_ds = dataset["test"].map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/928 [00:00<?, ? examples/s]

Map:   0%|          | 0/116 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

In [ ]:
import os
import shutil
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig

LOCAL_BASE = "/content/llama_3.1_8B"
DRIVE_BASE = "/content/drive/MyDrive/llama_3.1_8B"

LOCAL_CKPT = os.path.join(LOCAL_BASE, "checkpoints")
LOCAL_LORA = os.path.join(LOCAL_BASE, "lora_model")

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
DRIVE_LORA = os.path.join(DRIVE_BASE, "lora_model")

os.makedirs(LOCAL_CKPT, exist_ok=True)
os.makedirs(LOCAL_LORA, exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DRIVE_LORA, exist_ok=True)

print("Local:", LOCAL_BASE)
print("Drive:", DRIVE_BASE)

class BackupToDriveCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):

        ckpt_name = f"checkpoint-{state.global_step}"

        src = os.path.join(args.output_dir, ckpt_name)
        dst = os.path.join(DRIVE_CKPT, ckpt_name)

        if os.path.exists(src):
            shutil.copytree(dst=dst, src=src, dirs_exist_ok=True)
            print(f"☁️ Backed up: {ckpt_name}")


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    dataset_text_field="text",
    max_seq_length=1536,

    args=SFTConfig(
        output_dir=LOCAL_CKPT,

        # ========================
        # 🔥 CHECKPOINT SAVE
        # ========================
        save_strategy="steps",
        save_steps=25,
        save_total_limit=3,

        eval_strategy="steps",
        eval_steps=25,

        # ========================
        # TRAIN SETTING
        # ========================
        per_device_train_batch_size=1,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=6,

        logging_steps=50,
        bf16=False,
        fp16=True,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_steps=25,

        dataloader_num_workers=2,
        packing=True,
        gradient_checkpointing = True,

        report_to="wandb",
        run_name="llama-3.1-8b-lora-run-1",
    ),
)
trainer.add_callback(BackupToDriveCallback())

Local: /content/llama_3.1_8B
Drive: /content/drive/MyDrive/llama_3.1_8B


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/928 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/928 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/116 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=6):   0%|          | 0/116 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 137 | Num Epochs = 6 | Total steps = 210
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 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


wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


...

Unsloth: Restored added_tokens_decoder metadata in /content/llama_3.1_8B/checkpoints/checkpoint-200/tokenizer_config.json.


☁️ Backed up: checkpoint-200


Unsloth: Restored added_tokens_decoder metadata in /content/llama_3.1_8B/checkpoints/checkpoint-210/tokenizer_config.json.


☁️ Backed up: checkpoint-210


TrainOutput(global_step=210, training_loss=1.0784519513448079, metrics={'train_runtime': 6540.4501, 'train_samples_per_second': 0.126, 'train_steps_per_second': 0.032, 'total_flos': 5.532014026491494e+16, 'train_loss': 1.0784519513448079, 'epoch': 6.0})

In [ ]:
from google.colab import userdata

hf_token = userdata.get("HF_WRITE_TOKEN")

PATH_TO_SAVE = "/content/llama_3.1_8B/checkpoints/checkpoint-100"

model.save_pretrained(PATH_TO_SAVE)
tokenizer.save_pretrained(PATH_TO_SAVE)

model.push_to_hub_merged(
    "quyetdev/llama3.1_8B_fine_tuned_16bit",
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in quyetdev/llama3.1_8B_fine_tuned_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   2%|2         | 23.9MB / 1.17GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:21<00:00, 125.47s/it]


Unsloth: Merge process complete. Saved to `/content/quyetdev/llama3.1_8B_fine_tuned_16bit`
